# Introduction: Ollama LLM Backend

Ollama is a powerful open-source platform for running large language models (LLMs) locally on your machine. It operates as an **independent service** with a built-in server that exposes an **OpenAI-compatible API endpoint** at `http://localhost:11434/v1`. 

This makes Ollama perfect for AI agent development, local experimentation, and production deployments without cloud dependencies. You'll download models once, then query them through familiar OpenAI SDK calls — but everything runs offline on your hardware.

## Why Ollama?

- **Local-first**: No API costs, no data sent to third parties
- **OpenAI compatible**: Use existing Python SDKs and agent frameworks
- **Model management**: Easy pull/list/ps commands for model lifecycle
- **Performance optimized**: GPU acceleration, streaming, context caching
- **Developer friendly**: REST API + CLI in one package

Let's get Ollama running and start building!

In [31]:
import subprocess
import os
import signal
import time
import atexit

# Global for cleanup
OLLAMA_PROCESS = None

def start_ollama_server():
    global OLLAMA_PROCESS

    log_file = "ollama_server.log"

    # Kill any existing ollama processes
    subprocess.run(["pkill", "ollama"], capture_output=True)
    time.sleep(2)

    # Start ollama serve in background
    print("🚀 Starting Ollama server...")
    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=open(log_file, "w"),
        stderr=subprocess.STDOUT,
        preexec_fn=os.setpgrp,
    )
    print(f"📄 Server logs: {log_file}")
    print(f"📍 API endpoint: http://localhost:11434")

    # Wait for server to start
    print("⏳ Waiting 5 seconds for server startup...")
    time.sleep(5)

    # Store process PID for cleanup
    if OLLAMA_PROCESS is None:
        OLLAMA_PROCESS = process.pid
        print(f"✅ Ollama server ready! PID: {OLLAMA_PROCESS}")
    else:
        print("✅ Ollama server already running!")

    # Cleanup function
    def cleanup_ollama():
        global OLLAMA_PROCESS
        try:
            if OLLAMA_PROCESS is not None:
                os.killpg(os.getpgid(OLLAMA_PROCESS), signal.SIGTERM)
                print("🛑 Ollama server stopped")
                OLLAMA_PROCESS = None
        except Exception:
            pass

    atexit.register(cleanup_ollama)

def stop_ollama_server():
    global OLLAMA_PROCESS
    try:
        subprocess.run(["pkill", "ollama"], capture_output=True)
        print("🛑 Ollama server stopped (via pkill)")
        OLLAMA_PROCESS = None
    except Exception as e:
        print(f"⚠️ Error stopping Ollama: {e}")


# Call it once when this cell runs
start_ollama_server()

In [33]:
# Check if Ollama server is running (this should return "\NAME ID SIZE PROCESSOR CONTEXT UNTIL")
!ollama ps

]11;?\Error: could not connect to ollama server, run 'ollama serve' to start it


## Ollama Model Management

`ollama ps` to see running models

`ollama list` to see downloaded models

`ollama pull <modelname>` to download new models

**Naming convention**: `modelfamily:< # of weights in billions>b`
- `qwen3.5`: Model family name
- `0.8b`: 0.8 billion parameters (smaller = faster, less capable)
- Common sizes: `0.8b`, `2b`, `4b`, `9b`, `27b`

Smaller models run faster on consumer hardware but have less reasoning ability. Let's download a tiny-but-capable model.

### Model Storage Location
Ollama stores models in the directory specified by the **OLLAMA_MODELS** environment variable, by default this would store models somewhere in your home directory, but TACC limits your home directory storage quota to 10GB, so storing models elsewhere, like in the purgable SCRATCH filesystem is recommended. We have set your OLLAMA_MODELS variable to point to a shared library of predownloaded models that we prepared, run the code below to examine it's location and the models available.

In [28]:
# Check the OLLAMA_MODELS environment variable value
! echo $OLLAMA_MODELS

# Check the list of models already downloaded
! ollama list

/scratch/projects/tacc/ai_models/.ollama_models
]11;?\NAME                       ID              SIZE      MODIFIED       
qwen3.5:0.8b               f3817196d142    1.0 GB    35 minutes ago    
qwen2.5-coder:0.5b         4ff64a7f502a    397 MB    17 hours ago      
smollm2:135m               9077fe9d2ae1    270 MB    18 hours ago      
qwen2.5:1.5b               65ec06548149    986 MB    19 hours ago      
tinyllama:latest           2644915ede35    637 MB    20 hours ago      
qwen3:32b                  030ee887880f    20 GB     20 hours ago      
qwen3.5:35b                3460ffeede54    23 GB     3 days ago        
gemma3:27b                 a418f5838eaf    17 GB     3 days ago        
gemma3:12b                 f4031aab637d    8.1 GB    3 days ago        
gemma3:1b                  8648f39daa8f    815 MB    3 days ago        
qwen3.5:2b-50000           2593b544626e    2.7 GB    8 days ago        
qwen3.5:0.8b-50000         a114dc9d9ccb    1.0 GB    8 days ago        
qwen3.5:2b 

### Downloading your own models (optional)
If you want to try downloading testing out models yourself, you need to:
1) Set `USE_SHARED_MODELS=False` and run the code below to restart the ollama server and direct it to use your personal SCRATCH folder for storage
2) Browse ollama's online library: https://ollama.com/library?sort=popular and copy the name of a model you want to download and run `ollama pull modelname:<size>`

In [15]:
import os

# 📁 Choose your models directory
USE_SHARED_MODELS = True  # Set to False for personal $SCRATCH/.ollama_models

# Auto-detect SCRATCH (common on HPC systems like TACC)
SCRATCH = os.environ.get('SCRATCH', os.path.expanduser('~/scratch'))

if USE_SHARED_MODELS:
    models_dir = "/scratch/projects/tacc/ai_models/.ollama_models"
    print(f"🌐 Using shared models: {models_dir}")
else:
    models_dir = f"{SCRATCH}/.ollama_models"
    # Create personal directory if it doesn't exist
    os.makedirs(models_dir, exist_ok=True)
    print(f"👤 Using personal models: {models_dir}")

# 🔄 Restart Ollama server with new OLLAMA_MODELS environment
print(f"\n🔧 Setting OLLAMA_MODELS={models_dir} and restarting server...")
os.environ['OLLAMA_MODELS'] = models_dir

# Reuse the existing start_ollama_server() function from above 
start_ollama_server()

print(f"✅ Server restarted! Models will be stored in: {models_dir}")
print(f"📋 Check logs: ollama_server.log")


🌐 Using shared models: /scratch/projects/tacc/ai_models/.ollama_models

🔧 Setting OLLAMA_MODELS=/scratch/projects/tacc/ai_models/.ollama_models and restarting server...
🚀 Starting Ollama server...
📄 Server logs: ollama_server.log
📍 API endpoint: http://localhost:11434
⏳ Waiting 5 seconds for server startup...
✅ Ollama server already running!
✅ Server restarted! Models will be stored in: /scratch/projects/tacc/ai_models/.ollama_models
📋 Check logs: ollama_server.log


In [14]:
# Pull the qwen3.5:0.8b model (~500MB)
!ollama pull "qwen3.5:0.8b"

# Verify it's downloaded
!ollama list

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling manifest      ▏ 123 KB/1.0 GB                  
pulling manifest      ▏ 8.9 MB/1.0 GB                  
pulling manifest      ▏  22 MB/1.0 GB                  
pulling manifest      ▏  29 MB/1.0 GB                  
pulling manifest      ▏  44 MB/1.0 GB                  
pulling manifest      ▏  58 MB/1.0 GB                  
pulling manifest      ▏  66 MB/1.0 GB                  
pulling manifest      ▏  79 MB/1.0 GB                  
pulling manifest      ▏  94 MB/1.0 GB                  
pulling manifest      ▏ 101 MB/1.0 GB                  
pulling manifest      ▏ 115 MB/1.0 GB  115 MB/s      7s
pulling manifest      ▏ 129 MB/1.0 GB  115 MB/s      7s
pulling manifest      ▏ 137 MB/1.0 GB  115 MB/s      7s
pulling manifest      ▏ 150 MB/1.0 GB  115 MB/s      7s
pulling manifest   

Great, your new model is downloaded and ready to be used!  I'd recommend for now switching back to the shared model library and restarting ollama by setting USE_SHARED_MODELS=False above and rerunning the code block because we will be using some large models below that have already been downloaded for you. 

## Programtically Querying Ollama
I've written a function `generate_with_ollama` below that will allow us to send a prompt to ollama and return the generated text from the chosen llm as well as some performance metrics.  Running the block below will execute the function without arguments, which defaults to a prompt asking for an essay on fish.

### Explanation of Input Parameters

- **model** — The specific Ollama model and tag to use for generation (e.g., `"gemma3:1b"` for the 1-billion parameter Gemma 3 model).
- **system_prompt** — The initial system message that sets the AI's behavior and context (used only when `messages=None`).
- **user_prompt** — The user input prompt for single-turn conversations (used only when `messages=None`).
- **messages** — Optional list of message dictionaries for multi-turn conversations (e.g., `[{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]`. Overrides `system_prompt` and `user_prompt` when provided.
- **context_length** — Maximum token context window via `num_ctx` (`2048` tokens by default).
- **verbose** — Enable detailed stats output via `pprint` or JSON (`True`/`False`).
- **pretty_print** — Output format when `verbose=True` (`"pprint"` for formatted dict or `"json"`).
- **seed** — Random seed for reproducible generations (`-1` for random, Ollama default).
- **temperature** — Controls output randomness (`0.8` default; lower = more deterministic).

In [16]:
import requests
import json
import time
from pprint import pprint
import threading
import subprocess


def _monitor_vram(stop_event, interval=0.2):
    """Monitor VRAM usage with nvidia-smi and return peak in MiB."""
    peak_vram = 0
    while not stop_event.is_set():
        try:
            result = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                capture_output=True, text=True, check=True
            )
            usage_list = [int(v) for v in result.stdout.strip().splitlines() if v.strip()]
            if usage_list:
                peak_vram = max(peak_vram, max(usage_list))
        except Exception:
            pass
        time.sleep(interval)
    return peak_vram


def generate_with_ollama(
    model: str = "gemma3:1b",
    system_prompt: str = "You are a helpful AI assistant.",
    user_prompt: str = "Write a short essay about fish.",
    messages: list = None,
    context_length: int = 2048,
    verbose: bool = False,
    pretty_print: str = "pprint",  # "pprint" or "json"
    seed: int = -1,
    temperature: float = 0.8,
) -> dict:
    """
    Generate text using Ollama's /api/chat endpoint, and measure performance + peak VRAM usage.
    Supports multi-turn conversations via optional 'messages' parameter.
    """
    url = "http://localhost:11434/api/chat"

    # Determine messages for payload
    if messages is not None:
        payload_messages = messages
    else:
        payload_messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]

    payload = {
        "model": model,
        "messages": payload_messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": 0.9,
            "num_ctx": context_length,
            "seed": seed,
        },
    }

    # Start VRAM monitoring in a background thread
    stop_event = threading.Event()
    peak_vram = {"value": 0}

    def monitor():
        peak = _monitor_vram(stop_event)
        peak_vram["value"] = peak

    monitor_thread = threading.Thread(target=monitor, daemon=True)
    monitor_thread.start()

    # Measure wall time
    start_time = time.time()
    response = requests.post(url, json=payload)
    wall_time = time.time() - start_time

    # Stop monitoring
    stop_event.set()
    monitor_thread.join(timeout=1)

    ns_to_s = 1e-9

    if response.status_code == 200:
        result = response.json()
        stats = {
            "response": result["message"]["content"],
            "model": model,
            "context_length": f"{context_length} tokens",
            "total_duration": f"{result.get('total_duration', 0) * ns_to_s:.3f}s",
            "load_duration": f"{result.get('load_duration', 0) * ns_to_s:.3f}s",
            "prompt_eval_count": f"{result.get('prompt_eval_count', 0)} tokens",
            "prompt_eval_duration": f"{result.get('prompt_eval_duration', 0) * ns_to_s:.3f}s",
            "eval_count": f"{result.get('eval_count', 0)} tokens",
            "eval_duration": f"{result.get('eval_duration', 0) * ns_to_s:.3f}s",
            "wall_time": f"{wall_time:.3f}s",
            "tokens_per_second": f"{result.get('eval_count', 0) / max((result.get('eval_duration', 1) * ns_to_s), 0.001):.2f} tokens/s",
            "peak_vram": f"{peak_vram['value']} MiB",
            "seed": seed,
            "temperature": temperature,
        }
    else:
        stats = {
            "error": f"HTTP {response.status_code}: {response.text}",
            "model": model,
            "peak_vram": f"{peak_vram['value']} MiB",
            "seed": seed,
            "temperature": temperature,
        }

    if verbose:
        if pretty_print == "pprint":
            pprint(stats, indent=2, width=80, sort_dicts=False)
        elif pretty_print == "json":
            print(json.dumps(stats, indent=2))

    return stats

In [17]:
# Default Generation Test, writes an essay on fish with gemma3:1b
print("=== Default Generation Test ===")
response_data = generate_with_ollama(verbose=True)

=== Default Generation Test ===
{ 'response': "Okay, here's a short essay about fish, aiming for a balance of "
              'informative and evocative language. I’ve focused on capturing a '
              'sense of their beauty and importance. You can let me know if '
              'you’d like me to focus on a specific aspect (e.g., their '
              'evolution, their role in the ecosystem, etc.)!\n'
              '\n'
              '---\n'
              '\n'
              '**The Silent Symphony of the Deep**\n'
              '\n'
              'Fish. The very word conjures images of shimmering scales, swift '
              'movements, and a world beneath the surface. They are often '
              'overlooked, relegated to the periphery of our consciousness, '
              'yet they are undeniably vital to the health and vibrancy of our '
              'planet. More than just a collection of fins and gills, fish '
              'represent a remarkable and ancient evolutionary j

### Explaination of generation parameters

- **model** — The specific model and version used for text generation (e.g., `gemma3:1b` indicates the 1-billion parameter Gemma 3 model).
- **context_length** — The maximum number of tokens the model can consider in a single generation context, including both the prompt and the generated output.
- **total_duration** — The total elapsed time from the start of the generation request to completion, including loading, processing, and output.
- **load_duration** — The time taken to load the model into memory before generation begins.
- **prompt_eval_count** — The number of tokens in the input prompt evaluated by the model.
- **prompt_eval_duration** — The time spent processing or “evaluating” the input prompt tokens before generation starts.
- **eval_count** — The number of tokens generated by the model in the response.
- **eval_duration** — The total time taken to generate all output tokens.
- **wall_time** — The real-world clock time between the beginning and end of the complete generation process (usually close to `total_duration`).
- **tokens_per_second** — The generation speed, indicating how many tokens the model produced per second during output.
- **peak_vram** — The maximum GPU memory used during the generation process.

### Function to display Markdown formatted model responses
Here is a small utility function I've written to display the model responses in a readable HTML panel with rendered markdown formatting and code block highlighting.

In [18]:
from IPython.display import HTML, display
from markdown import markdown
import re

def display_markdown_panel(response_text, title="Model Response"):
    """
    Render markdown with syntax-highlighted code blocks in a styled panel
    """
    # Convert markdown to HTML
    html_content = markdown(response_text, extensions=['fenced_code', 'codehilite'])
    
    # Add custom CSS for code blocks if not already highlighted
    enhanced_html = re.sub(
        r'<pre><code([^>]*)>(.*?)</code></pre>',
        r'<pre><code\1 class="python">\2</code></pre>',
        html_content,
        flags=re.DOTALL
    )
    
    panel_html = f"""
    <style>
    .response-panel {{
        border: 2px solid #4a90e2;
        border-radius: 12px;
        padding: 20px;
        margin: 10px 0;
        background: linear-gradient(145deg, #f8f9ff, #ffffff);
        box-shadow: 0 4px 12px rgba(74, 144, 226, 0.15);
        font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    }}
    .response-panel h1, .response-panel h2, .response-panel h3 {{
        color: #4a90e2;
        border-bottom: 2px solid #e1e8ff;
        padding-bottom: 8px;
    }}
    .response-panel pre {{
        background: #2d3748;
        border-radius: 8px;
        padding: 16px;
        overflow-x: auto;
        margin: 16px 0;
        border: 1px solid #4a5568;
    }}
    .response-panel code {{
        background: #f7fafc;
        padding: 2px 6px;
        border-radius: 4px;
        font-family: 'Monaco', 'Menlo', 'Ubuntu Mono', monospace;
    }}
    </style>
    <div class="response-panel">
        <h3 style="margin: 0 0 15px 0;">{title}</h3>
        <div style="line-height: 1.6; color: #333;">{enhanced_html}</div>
    </div>
    """
    display(HTML(panel_html))

### Multi-turn Conversation Generation Example
Multi-turn chatbot style conversations are communicated to the model by sending a `messages` object in our api request that lists the past system, user, and assistant response messages

In [19]:
# ===== Multi-turn conversation example =====

# Build messages with our conversation history
messages = [
    {"role": "system", "content": "You are a Python programming expert."},
    {"role": "user", "content": "What is a list comprehension in Python?"},
    {"role": "assistant", "content": "A list comprehension is a concise way to create lists using the syntax: [expression for item in iterable if condition]."},
    {"role": "user", "content": "Can you give me an example that filters even numbers from a range?"}
]

# generate the next response from the assistant
response_data = generate_with_ollama(
    messages=messages,
)

# print out the response
display_markdown_panel(response_data['response'], title="Multi-turn Conversation Response")

## Context Length in Ollama

**Context length** is the maximum number of tokens (words/subwords) a model can process in memory at once. It includes your system prompt, conversation history, current input, and generated response. Ollama often defaults to 2048-8192 tokens based on VRAM, but models like Gemma support much larger contexts (32k+). It is important to be aware of your context length limit as backends will often silently truncate parts of the prompt or entire messages to fit the remaining text it into the model's context window, often without notifying you that it has done so.  In the example below, we will append a veritable tide of llama emojis around our prompt to see how ollama handles prompt truncation.

In [20]:
# Fill system prompt with 4000 llama emojis to test context
llama_emoji = "🐐"  # Llama emoji
distracting_llamas = llama_emoji * 4000

print(f"📏 Length of distracting llama herd: {len(distracting_llamas)} 🐐s")
print(f"{distracting_llamas[:50]}...")
      
# Generate a haiku but with llamas at the beginning of the prompt
print("\n\n" + "="*20 + "\n" + "Llamas before the user prompt\n" + "="*20)
response = generate_with_ollama(
    context_length = 2048,
    seed = 2,
    user_prompt= distracting_llamas + "Write a haiku about vines",
    verbose = True
)

# Generate a haiku but with llamas at the end of the prompt
print("\n\n" + "="*20 + "\n" + "Llamas after the user prompt\n" + "="*20)
response = generate_with_ollama(
    context_length = 2048,
    seed = 2,
    user_prompt="Write a haiku about vines" + distracting_llamas,
    verbose = True
)

# Generate a haiku but with llamas in a message before the prompt
print("\n\n" + "="*20 + "\n" + "Llamas in their own separate user message before the user prompt\n" + "="*20)
# Build messages with our conversation history
messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user", "content": distracting_llamas},
    {"role": "user", "content": "Write a haiku about vines"},
]
response = generate_with_ollama(
    context_length = 2048,
    seed = 2,
    messages = messages,
    verbose = True
)

📏 Length of distracting llama herd: 4000 🐐s
🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐🐐...


Llamas before the user prompt
{ 'response': 'That’s a fantastic and wonderfully complex list! It really '
              'captures a sense of vastness and interconnectedness. Do you '
              'want me to do anything with it? For example, would you like me '
              'to:\n'
              '\n'
              '*   **Analyze the list?** (e.g., identify patterns, themes?)\n'
              '*   **Generate a related list?** (e.g., a list of things that '
              'could be represented by this?)\n'
              '*   **Expand on a specific element?** (e.g., a particular word '
              'or phrase?)\n'
              '*   **Just respond to your prompt?** (e.g., “What’s the most '
              'interesting part of this list?”)',
  'model': 'gemma3:1b',
  'context_length': '2048 tokens',
  'total_duration': '0.865s',
  'load_duration': '0.225s',
  'prompt_eval_count': '2048 toke

The model only understood that we wanted a haiku in the third example where we put the distracting llamas in their own individual user message, so it appears that by default ollama is removing long messages entirely from the conversation!  Notice how there were no error message in either case. Each LLM backend will handle truncation differently, so it's important to be aware of this common failure mode, particularly in agentic systems where long prompts are generated at runtime programtically.

## Scaling model size to match prompt difficulty
Larger models can more reliably follow more instructions and constraints from your prompt than small models. To demonstrate this, I have prepared an easy and a hard prompt for an LLM, where the LLM is going to try and produce HTML code that when rendered looks like a house or a city of houses.  This is a particularly difficult task for LLMs which struggle with spatial reasoning (generating 2D grids with shapes drawn in them, building layouts etc.).

**Easy Prompt**
The easy prompt in the example below provides an HTML snippet in the prompt that when rendered produces a little llama house with a dialog bubble that shows up when you mouse over it.  Our prompt asks the LLM to reproduce this snippet but with different colors and a new dialog. 

**Hard Prompt**
The hard prompt also provides an HTML snippet with an stacked apartment complex of llama abodes, and asks the LLM to generating a randomized layout of houses and apartments with new colors/dialogs.  This is particularly challenging because the LLM is now mixing houses and apartments that have different vertical sizes, and it is likely to get confused about how to align them vertically in the output.

You can modify `user_prompt_corrections` to add in your own additional instructions below.  Uncomment the hard mode prompt to try generating a city. Try swapping to larger models to see which prompt each model is able to handle (note that it might take a minute to load the larger models for your first generation attempt).  If you don't see anything rendered, try uncommenting the debug info print out at the end and rerunning to see the full model response text.

In [21]:
# Let's load an easy prompt that requests the LLM generate a llama in a house with a witty popup text
with open('llama_house_prompt.txt', 'r') as f:
    prompt_text = f.read()

# # Uncomment for Hard Mode: load a prompt that requests the LLM generate a whole city of llamas
# with open('llama_city_prompt.txt', 'r') as f:
#     prompt_text = f.read()

# Add any additional instructions to the prompt here 
user_prompt_corrections = ""

response_data = generate_with_ollama(
    user_prompt=prompt_text + "Additional Instructions: " + user_prompt_corrections, 
    context_length=30000, 
    verbose=False, 
    model = "gemma3:1b"
    # model = "gemma3:27b"
    # model="gpt-oss:120b"
)

# Extract HTML content from <pre><code> block and render
from bs4 import BeautifulSoup
from IPython.display import display, HTML
import re

raw_response = response_data['response']

# Extract content from first <pre><code> block
soup = BeautifulSoup(raw_response, 'html.parser')
pre_code = soup.find('pre')
if pre_code:
    code_content = pre_code.find('code')
    html_content = code_content.get_text() if code_content else pre_code.get_text()
else:
    # Fallback: try regex extraction from ```html blocks
    html_match = re.search(r'```html\s*(.*?)```', raw_response, re.DOTALL)
    if html_match:
        html_content = html_match.group(1)
    else:
        html_content = raw_response

print("HTML-rendered response from code block:")
display(HTML(html_content))

# Debug info
# print(f"\nExtracted HTML length: {len(html_content)} characters")
# print("Preview (first 300 chars):")
# print(html_content[:300] + "..." if len(html_content) > 300 else html_content)


HTML-rendered response from code block:


## Cleanup (Run Before Shutting Down)

```python
# Stop Ollama server using stored PID
cleanup_ollama()


In [32]:
stop_ollama_server()

🛑 Ollama server stopped (via pkill)
